In [1]:
# 1. نصب کتابخانه‌های ضروری (در صورت نیاز محیط اجرا)
!pip install -q transformer-lens torch numpy scipy

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 5.8 MB/s eta 0:00:00


In [2]:
# 2. بررسی اتصال سخت‌افزاری و راه‌اندازی اولیه
import torch
import warnings

warnings.filterwarnings("ignore")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Runtime Ready. Device: {device.upper()}")

if device == "cuda":
    print(f"🎮 GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"🧠 VRAM Available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ WARNING: Running on CPU. Mechanistic interventions will be slow.")

✅ Runtime Ready. Device: CUDA
🎮 GPU Model: Tesla T4
🧠 VRAM Available: 15.64 GB


In [3]:
import torch
import numpy as np
import json
import warnings
from dataclasses import dataclass, asdict
from typing import Dict, List, Tuple, Optional, Any
from scipy.stats import norm
from transformer_lens import HookedTransformer

warnings.filterwarnings("ignore", category=UserWarning)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

MODEL_NAME = "Qwen/Qwen2.5-0.5B"


# ==============================================================================
# 0. SERIALIZATION & HELPERS
# ==============================================================================

def to_jsonable(obj: Any) -> Any:
    if isinstance(obj, dict):
        return {str(k): to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_jsonable(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, np.generic):
        return obj.item()
    if isinstance(obj, torch.Tensor):
        return obj.item() if obj.ndim == 0 else obj.detach().cpu().tolist()
    if isinstance(obj, (str, int, float, bool)) or obj is None:
        return obj
    if hasattr(obj, "__dataclass_fields__"):
        return to_jsonable(asdict(obj))
    return str(obj)


def extract_context_family(ctx_str: str) -> str:
    for prefix in ["train_", "calib_", "test_", "ood_"]:
        if ctx_str.startswith(prefix):
            return ctx_str[len(prefix):]
    return ctx_str


# ==============================================================================
# 1. SEQUENTIAL BAYESIAN SCM WITH DECISION-ENTROPY ENGINE
# ==============================================================================

@dataclass
class TargetSpec:
    layer: int
    component: str
    index: int = 0
    keep_ratio: float = 0.0

    @property
    def dose(self) -> float:
        return float(np.clip(1.0 - self.keep_ratio, 0.0, 1.0))

    @property
    def key(self) -> str:
        return f"L{self.layer}_{self.component}_{self.index}"


@dataclass
class PromptItem:
    prompt: str
    target_str: str
    distractor_str: str


class DecisionBayesianSelfGraph:
    def __init__(self, prior_variance: float = 0.5, aleatoric_variance: float = 0.005):
        self.prior_variance = prior_variance
        self.aleatoric_variance = aleatoric_variance
        self.component_models: Dict[str, Dict[str, Dict[str, Any]]] = {}
        self.thresh_gain = 0.005
        self.thresh_hazard = -0.05

    def init_node(self, target_key: str, ctx_str: str):
        family = extract_context_family(ctx_str)
        if target_key not in self.component_models:
            self.component_models[target_key] = {}
        if family not in self.component_models[target_key]:
            mu_0 = np.zeros(2)
            prec_0 = np.eye(2) / self.prior_variance
            self.component_models[target_key][family] = {
                "mu": mu_0,
                "precision": prec_0,
                "cov": np.linalg.inv(prec_0),
                "n": 0
            }

    def compute_decision_entropy(self, mu: float, sigma: float) -> Tuple[float, np.ndarray]:
        sigma = max(sigma, 1e-6)
        p_hazard = norm.cdf((self.thresh_hazard - mu) / sigma)
        p_gain = 1.0 - norm.cdf((self.thresh_gain - mu) / sigma)
        p_zero = max(0.0, 1.0 - p_hazard - p_gain)

        probs = np.array([p_gain, p_zero, p_hazard])
        probs = np.clip(probs, 1e-12, 1.0)
        probs = probs / np.sum(probs)
        entropy = -float(np.sum(probs * np.log(probs)))
        return entropy, probs

    def update(self, target_key: str, ctx_str: str, dose: float, delta_obs: float) -> float:
        family = extract_context_family(ctx_str)
        self.init_node(target_key, ctx_str)
        node = self.component_models[target_key][family]

        x = np.array([dose, dose**2])
        prec_old = node["precision"]
        mu_old = node["mu"]

        inv_noise = 1.0 / self.aleatoric_variance
        prec_new = prec_old + inv_noise * np.outer(x, x)
        cov_new = np.linalg.inv(prec_new)
        mu_new = cov_new @ (prec_old @ mu_old + inv_noise * x * delta_obs)

        det_old = np.linalg.det(prec_old)
        det_new = np.linalg.det(prec_new)
        ig = float(0.5 * np.log(max(det_new / det_old, 1e-12)))

        node["precision"] = prec_new
        node["cov"] = cov_new
        node["mu"] = mu_new
        node["n"] += 1
        return ig

    def predict(self, targets: List[TargetSpec], ctx_str: str) -> Tuple[float, float]:
        family = extract_context_family(ctx_str)
        pred_delta = 0.0
        total_var = 0.0

        for t in targets:
            d = t.dose
            if d == 0.0:
                continue
            x = np.array([d, d**2])
            self.init_node(t.key, ctx_str)
            node = self.component_models[t.key][family]
            pred_delta += float(x @ node["mu"])
            total_var += float(x @ node["cov"] @ x)

        return pred_delta, total_var


# ==============================================================================
# 2. MECHANISTIC SENSORS INTERFACE
# ==============================================================================

class MechanisticSensorInterface:
    def __init__(self, model_name: str = MODEL_NAME, device: str = "cuda"):
        self.device = device if torch.cuda.is_available() else "cpu"
        print(f"🔄 Initializing Sensors ({model_name}) on {self.device.upper()}...")
        self.model = HookedTransformer.from_pretrained(
            model_name, device=self.device, torch_dtype=torch.float32, default_padding_side="left"
        )
        self.model.eval()
        for p in self.model.parameters():
            p.requires_grad_(False)

    def _build_hooks(self, targets: List[TargetSpec]):
        hooks = []
        for t in targets:
            keep = t.keep_ratio
            if t.component == "head":
                h_idx = t.index
                def head_hook_fn(act, hook, h=h_idx, r=keep):
                    act[:, -1, h, :] = act[:, -1, h, :] * r
                    return act
                hooks.append((f"blocks.{t.layer}.attn.hook_z", head_hook_fn))
            elif t.component == "mlp":
                def mlp_hook_fn(act, hook, r=keep):
                    act[:, -1, :] = act[:, -1, :] * r
                    return act
                hooks.append((f"blocks.{t.layer}.hook_mlp_out", mlp_hook_fn))
        return hooks

    def _resolve_token_id(self, token_str: str) -> int:
        toks = self.model.to_tokens(token_str, prepend_bos=False)[0]
        return int(toks[-1].item())

    @torch.no_grad()
    def measure_grounded_outcomes(self, items: List[PromptItem], targets: Optional[List[TargetSpec]] = None) -> float:
        prompts = [it.prompt for it in items]
        tokens = self.model.to_tokens(prompts)
        batch_size = len(items)
        b_idx = torch.arange(batch_size, device=self.device)

        target_ids = torch.tensor([self._resolve_token_id(it.target_str) for it in items], device=self.device)
        distractor_ids = torch.tensor([self._resolve_token_id(it.distractor_str) for it in items], device=self.device)

        clean_logits = self.model(tokens)[:, -1, :].float()
        clean_m = clean_logits[b_idx, target_ids] - clean_logits[b_idx, distractor_ids]

        if not targets:
            return 0.0

        with self.model.hooks(fwd_hooks=self._build_hooks(targets)):
            int_logits = self.model(tokens)[:, -1, :].float()

        int_m = int_logits[b_idx, target_ids] - int_logits[b_idx, distractor_ids]
        return float(torch.mean(int_m - clean_m).item())


# ==============================================================================
# 3. EXPERIMENT DESIGN ENGINES
# ==============================================================================

class ActiveExperimentEngine:
    def __init__(self, scm: DecisionBayesianSelfGraph):
        self.scm = scm
        self.candidates = np.linspace(0.05, 0.40, 8)

    def select_ig_active_dose(self, target_key: str, ctx_str: str, executed_doses: List[float]) -> float:
        self.scm.init_node(target_key, ctx_str)
        family = extract_context_family(ctx_str)
        node = self.scm.component_models[target_key][family]
        best_score, best_d = -float("inf"), self.candidates[0]

        for d in self.candidates:
            if any(abs(d - ed) < 0.03 for ed in executed_doses):
                continue
            x = np.array([d, d**2])
            prec_sim = node["precision"] + (1.0 / self.scm.aleatoric_variance) * np.outer(x, x)
            ig = 0.5 * np.log(max(np.linalg.det(prec_sim) / np.linalg.det(node["precision"]), 1e-12))

            p_mu = float(x @ node["mu"])
            risk = max(0.0, -p_mu)
            score = ig / (float(d**2) + 2.0 * risk + 0.05)
            if score > best_score:
                best_score, best_d = score, float(d)
        return best_d

    def select_decision_active_dose(self, target: TargetSpec, ctx_str: str, executed_doses: List[float]) -> Tuple[float, bool]:
        self.scm.init_node(target.key, ctx_str)
        family = extract_context_family(ctx_str)
        node = self.scm.component_models[target.key][family]

        x_full = np.array([1.0, 1.0])
        mu_full = float(x_full @ node["mu"])
        sigma_full = float(np.sqrt(x_full @ node["cov"] @ x_full))
        curr_h, curr_probs = self.scm.compute_decision_entropy(mu_full, sigma_full)

        if sigma_full <= 0.45 and np.max(curr_probs) >= 0.92:
            return 0.0, True

        best_score, best_d = -float("inf"), self.candidates[0]

        for d in self.candidates:
            if any(abs(d - ed) < 0.03 for ed in executed_doses):
                continue
            x = np.array([d, d**2])

            prec_sim = node["precision"] + (1.0 / self.scm.aleatoric_variance) * np.outer(x, x)
            cov_sim = np.linalg.inv(prec_sim)
            sigma_full_sim = float(np.sqrt(x_full @ cov_sim @ x_full))

            sim_h, _ = self.scm.compute_decision_entropy(mu_full, sigma_full_sim)
            delta_h_decision = max(0.0, curr_h - sim_h)

            ig = 0.5 * np.log(max(np.linalg.det(prec_sim) / np.linalg.det(node["precision"]), 1e-12))

            pred_mu_d = float(x @ node["mu"])
            harm = max(0.0, -pred_mu_d)
            barrier_penalty = float(np.exp(harm * 40.0) - 1.0)

            score = (delta_h_decision * 3.0 + 0.3 * ig) / (float(d**2) + barrier_penalty + 0.05)

            if score > best_score:
                best_score, best_d = score, float(d)

        return best_d, False


# ==============================================================================
# 4. BENCHMARK CONTROLLER (MILESTONE 17)
# ==============================================================================

def run_milestone_17():
    sensors = MechanisticSensorInterface()
    corpus = {
        "ood_syntactic_agreement": [
            PromptItem("The keys to the cabinet are kept in the", " drawer", " drawers"),
            PromptItem("The books on the wooden table belong to the", " library", " libraries"),
            PromptItem("The pictures of the old palace show the ancient", " king", " kings"),
            PromptItem("The report about the recent discoveries was presented by", " researcher", " researchers"),
        ]
    }
    eval_ctx = "ood_syntactic_agreement"
    items = corpus[eval_ctx]

    benchmark_nodes = [
        {"desc": "High-Impact Sensitive MLP", "target": TargetSpec(12, "mlp", 0, keep_ratio=0.0)},
        {"desc": "Low-Impact Benign Head", "target": TargetSpec(4, "head", 0, keep_ratio=0.0)}
    ]

    max_budget = 3
    fixed_schedule = [0.10, 0.20, 0.35]
    random_schedules = [[0.08, 0.38, 0.15], [0.22, 0.06, 0.31], [0.35, 0.12, 0.25]]

    def resolve_governor(mu: float, sigma: float) -> str:
        if sigma > 0.50:
            return "VETO_EPISTEMIC"
        if mu < -0.05:
            return "VETO_HAZARD"
        if mu < 0.005:
            return "VETO_ZERO_GAIN"
        return "APPROVED"

    print("\n" + "=" * 155)
    print("🔬 MILESTONE 17: DECISION-AWARE ACTIVE SELF-DISCOVERY BENCHMARK")
    print("   Protocol: Strictly Blind Evaluation | Metric: Value of Information (VoI) vs Parameter Information Gain")
    print("=" * 155)

    comparative_results = []

    for b_node in benchmark_nodes:
        target = b_node["target"]
        desc = b_node["desc"]
        print(f"\n🎯 AUDITING NODE: {target.key} ({desc})")

        # 1. FIXED POLICY
        scm_fixed = DecisionBayesianSelfGraph()
        f_risk, f_ig = 0.0, 0.0
        for d in fixed_schedule:
            probe = TargetSpec(target.layer, target.component, target.index, keep_ratio=1.0 - d)
            obs = sensors.measure_grounded_outcomes(items, targets=[probe])
            f_risk += max(0.0, -obs)
            f_ig += scm_fixed.update(target.key, eval_ctx, d, obs)
        p_f, v_f = scm_fixed.predict([target], eval_ctx)

        # 2. RANDOM POLICY (N=3)
        r_risks, r_sigs, r_preds = [], [], []
        for r_sched in random_schedules:
            scm_rnd = DecisionBayesianSelfGraph()
            r_r = 0.0
            for d in r_sched:
                probe = TargetSpec(target.layer, target.component, target.index, keep_ratio=1.0 - d)
                obs = sensors.measure_grounded_outcomes(items, targets=[probe])
                r_r += max(0.0, -obs)
                scm_rnd.update(target.key, eval_ctx, d, obs)
            p_r, v_r = scm_rnd.predict([target], eval_ctx)
            r_risks.append(r_r)
            r_sigs.append(float(np.sqrt(v_r)))
            r_preds.append(p_r)

        # 3. IG-ACTIVE POLICY
        scm_ig = DecisionBayesianSelfGraph()
        engine_ig = ActiveExperimentEngine(scm_ig)
        ig_doses, ig_risk, ig_total = [], 0.0, 0.0
        for _ in range(max_budget):
            d = engine_ig.select_ig_active_dose(target.key, eval_ctx, ig_doses)
            ig_doses.append(d)
            probe = TargetSpec(target.layer, target.component, target.index, keep_ratio=1.0 - d)
            obs = sensors.measure_grounded_outcomes(items, targets=[probe])
            ig_risk += max(0.0, -obs)
            ig_total += scm_ig.update(target.key, eval_ctx, d, obs)
        p_ig, v_ig = scm_ig.predict([target], eval_ctx)

        # 4. DECISION-ACTIVE POLICY
        scm_dec = DecisionBayesianSelfGraph()
        engine_dec = ActiveExperimentEngine(scm_dec)
        dec_doses, dec_risk, dec_total_ig = [], 0.0, 0.0
        stopped_early = False

        for step in range(max_budget):
            d, should_stop = engine_dec.select_decision_active_dose(target, eval_ctx, dec_doses)
            if should_stop:
                stopped_early = True
                break
            dec_doses.append(d)
            probe = TargetSpec(target.layer, target.component, target.index, keep_ratio=1.0 - d)
            obs = sensors.measure_grounded_outcomes(items, targets=[probe])
            dec_risk += max(0.0, -obs)
            dec_total_ig += scm_dec.update(target.key, eval_ctx, d, obs)

        p_dec, v_dec = scm_dec.predict([target], eval_ctx)

        # Hidden Validation (Phase C)
        gt_full_delta = sensors.measure_grounded_outcomes(items, targets=[target])

        comparative_results.append({
            "target": target.key,
            "desc": desc,
            "gt_delta": gt_full_delta,
            "gt_verdict": resolve_governor(gt_full_delta, 0.0),
            "fixed": {
                "doses": fixed_schedule,
                "risk": f_risk,
                "sigma": float(np.sqrt(v_f)),
                "mae": abs(p_f - gt_full_delta),
                "verdict": resolve_governor(p_f, float(np.sqrt(v_f)))
            },
            "random": {
                "doses": "[Stochastic]",
                "risk": float(np.mean(r_risks)),
                "sigma": float(np.mean(r_sigs)),
                "mae": abs(float(np.mean(r_preds)) - gt_full_delta),
                "verdict": resolve_governor(float(np.mean(r_preds)), float(np.mean(r_sigs)))
            },
            "ig_active": {
                "doses": [round(x, 2) for x in ig_doses],
                "risk": ig_risk,
                "sigma": float(np.sqrt(v_ig)),
                "mae": abs(p_ig - gt_full_delta),
                "verdict": resolve_governor(p_ig, float(np.sqrt(v_ig)))
            },
            "decision_active": {
                "doses": [round(x, 2) for x in dec_doses],
                "risk": dec_risk,
                "sigma": float(np.sqrt(v_dec)),
                "mae": abs(p_dec - gt_full_delta),
                "verdict": resolve_governor(p_dec, float(np.sqrt(v_dec))),
                "stopped_early": stopped_early,
                "steps": len(dec_doses)
            }
        })

    # ==============================================================================
    # 5. AUDIT REPORT
    # ==============================================================================
    print("\n" + "=" * 155)
    print("📊 BENCHMARK 17: DECISION-AWARE VS. CLASSIC ACTIVE DISCOVERY REPORT")
    print("-" * 155)
    print(f"{'Target Key':<12} | {'Policy':<15} | {'Dose Sequence':<23} | {'Steps':<5} | {'Risk Incurred':<14} | {'Final σ':<8} | {'MAE':<7} | {'Verdict'}")
    print("-" * 155)

    for r in comparative_results:
        t_key = r["target"]

        f = r["fixed"]
        print(f"{t_key:<12} | {'FIXED':<15} | {str(f['doses']):<23} | {3:<5} | {f['risk']:<14.4f} | {f['sigma']:<8.3f} | {f['mae']:.4f} | {f['verdict']}")

        rnd = r["random"]
        print(f"{'':<12} | {'RANDOM':<15} | {rnd['doses']:<23} | {3:<5} | {rnd['risk']:<14.4f} | {rnd['sigma']:<8.3f} | {rnd['mae']:.4f} | {rnd['verdict']}")

        ig = r["ig_active"]
        print(f"{'':<12} | {'IG-ACTIVE (M16)':<15} | {str(ig['doses']):<23} | {3:<5} | {ig['risk']:<14.4f} | {ig['sigma']:<8.3f} | {ig['mae']:.4f} | {ig['verdict']}")

        dec = r["decision_active"]
        early_tag = " (STOP)" if dec["stopped_early"] else ""
        print(f"{'':<12} | {'DECISION-ACTIVE':<15} | {str(dec['doses']) + early_tag:<23} | {dec['steps']:<5} | {dec['risk']:<14.4f} | {dec['sigma']:<8.3f} | {dec['mae']:.4f} | {dec['verdict']}")
        print("-" * 155)

    with open("m17_decision_active_results.json", "w", encoding="utf-8") as f:
        json.dump(to_jsonable(comparative_results), f, indent=2)
    print("📁 Benchmark data exported to 'm17_decision_active_results.json'.")


if __name__ == "__main__":
    run_milestone_17()

🔄 Initializing Sensors (Qwen/Qwen2.5-0.5B) on CUDA...


config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Loaded pretrained model Qwen/Qwen2.5-0.5B into HookedTransformer

🔬 MILESTONE 17: DECISION-AWARE ACTIVE SELF-DISCOVERY BENCHMARK
   Protocol: Strictly Blind Evaluation | Metric: Value of Information (VoI) vs Parameter Information Gain

🎯 AUDITING NODE: L12_mlp_0 (High-Impact Sensitive MLP)

🎯 AUDITING NODE: L4_head_0 (Low-Impact Benign Head)

📊 BENCHMARK 17: DECISION-AWARE VS. CLASSIC ACTIVE DISCOVERY REPORT
-----------------------------------------------------------------------------------------------------------------------------------------------------------
Target Key   | Policy          | Dose Sequence           | Steps | Risk Incurred  | Final σ  | MAE     | Verdict
-----------------------------------------------------------------------------------------------------------------------------------------------------------
L12_mlp_0    | FIXED           | [0.1, 0.2, 0.35]        | 3     | 0.0578         | 0.491    | 0.0293 | VETO_HAZARD
             | RANDOM          | [Stochastic]  

In [ ]:
import torch
import numpy as np
import json
import warnings
from dataclasses import dataclass, asdict
from typing import Dict, List, Tuple, Optional, Any
from scipy.stats import norm
from transformer_lens import HookedTransformer

warnings.filterwarnings("ignore", category=UserWarning)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

MODEL_NAME = "Qwen/Qwen2.5-0.5B"


# ==============================================================================
# 0. SERIALIZATION & HELPERS
# ==============================================================================

def to_jsonable(obj: Any) -> Any:
    if isinstance(obj, dict):
        return {str(k): to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_jsonable(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, np.generic):
        return obj.item()
    if isinstance(obj, torch.Tensor):
        return obj.item() if obj.ndim == 0 else obj.detach().cpu().tolist()
    if isinstance(obj, (str, int, float, bool)) or obj is None:
        return obj
    if hasattr(obj, "__dataclass_fields__"):
        return to_jsonable(asdict(obj))
    return str(obj)


def extract_context_family(ctx_str: str) -> str:
    for prefix in ["train_", "calib_", "test_", "ood_"]:
        if ctx_str.startswith(prefix):
            return ctx_str[len(prefix):]
    return ctx_str


# ==============================================================================
# 1. ACTION-SENSITIVE UTILITY MATRIX (4 ACTIONS)
# ==============================================================================

class EpistemicActionUtility:
    THRESH_GAIN = 0.005
    THRESH_HAZARD = -0.05
    ALL_ACTIONS = ["APPROVED", "VETO_ZERO_GAIN", "VETO_HAZARD", "VETO_EPISTEMIC"]

    @staticmethod
    def get_utility(action: str, delta_m: float) -> float:
        if delta_m >= EpistemicActionUtility.THRESH_GAIN:
            if action == "APPROVED": return float(delta_m)
            if action == "VETO_EPISTEMIC": return -0.01  # جریمه فلج محافظه‌کارانه
            if action == "VETO_ZERO_GAIN": return -0.02
            if action == "VETO_HAZARD": return -0.03

        elif delta_m < EpistemicActionUtility.THRESH_HAZARD:
            if action == "VETO_HAZARD": return 0.05
            if action == "VETO_EPISTEMIC": return 0.03   # احتیاط موجه
            if action == "VETO_ZERO_GAIN": return 0.01
            if action == "APPROVED": return float(10.0 * delta_m)

        else:
            if action == "VETO_ZERO_GAIN": return 0.005
            if action == "VETO_EPISTEMIC": return 0.002
            if action == "VETO_HAZARD": return 0.0
            if action == "APPROVED": return float(delta_m)

        return -1.0

    @staticmethod
    def get_omniscient_oracle(delta_m: float) -> Tuple[str, float]:
        utils = [EpistemicActionUtility.get_utility(a, delta_m) for a in EpistemicActionUtility.ALL_ACTIONS]
        best_idx = int(np.argmax(utils))
        return EpistemicActionUtility.ALL_ACTIONS[best_idx], utils[best_idx]


# ==============================================================================
# 2. BAYESIAN SCM ENGINE
# ==============================================================================

@dataclass
class TargetSpec:
    layer: int
    component: str
    index: int = 0
    keep_ratio: float = 0.0

    @property
    def dose(self) -> float:
        return float(np.clip(1.0 - self.keep_ratio, 0.0, 1.0))

    @property
    def key(self) -> str:
        return f"L{self.layer}_{self.component}_{self.index}"


@dataclass
class PromptItem:
    prompt: str
    target_str: str
    distractor_str: str


class BayesianSelfGraph:
    def __init__(self, prior_variance: float = 0.5, aleatoric_variance: float = 0.005):
        self.prior_variance = prior_variance
        self.aleatoric_variance = aleatoric_variance
        self.component_models: Dict[str, Dict[str, Dict[str, Any]]] = {}

    def init_node(self, target_key: str, ctx_str: str):
        family = extract_context_family(ctx_str)
        if target_key not in self.component_models:
            self.component_models[target_key] = {}
        if family not in self.component_models[target_key]:
            self.component_models[target_key][family] = {
                "mu": np.zeros(2),
                "precision": np.eye(2) / self.prior_variance,
                "cov": np.eye(2) * self.prior_variance,
                "n": 0
            }

    def update(self, target_key: str, ctx_str: str, dose: float, delta_obs: float):
        family = extract_context_family(ctx_str)
        self.init_node(target_key, ctx_str)
        node = self.component_models[target_key][family]

        x = np.array([dose, dose**2])
        prec_old = node["precision"]
        mu_old = node["mu"]

        inv_noise = 1.0 / self.aleatoric_variance
        prec_new = prec_old + inv_noise * np.outer(x, x)
        cov_new = np.linalg.inv(prec_new)
        mu_new = cov_new @ (prec_old @ mu_old + inv_noise * x * delta_obs)

        node["precision"] = prec_new
        node["cov"] = cov_new
        node["mu"] = mu_new
        node["n"] += 1

    def predict(self, targets: List[TargetSpec], ctx_str: str) -> Tuple[float, float]:
        family = extract_context_family(ctx_str)
        pred_delta = 0.0
        total_var = 0.0

        for t in targets:
            if t.dose == 0.0:
                continue
            x = np.array([t.dose, t.dose**2])
            self.init_node(t.key, ctx_str)
            node = self.component_models[t.key][family]
            pred_delta += float(x @ node["mu"])
            total_var += float(x @ node["cov"] @ x)

        return pred_delta, total_var


# ==============================================================================
# 3. MECHANISTIC SENSORS INTERFACE
# ==============================================================================

class MechanisticSensorInterface:
    def __init__(self, model_name: str = MODEL_NAME, device: str = "cuda"):
        self.device = device if torch.cuda.is_available() else "cpu"
        print(f"🔄 Initializing Sensors ({model_name}) on {self.device.upper()}...")
        self.model = HookedTransformer.from_pretrained(
            model_name, device=self.device, torch_dtype=torch.float32, default_padding_side="left"
        )
        self.model.eval()
        for p in self.model.parameters():
            p.requires_grad_(False)

    def _build_hooks(self, targets: List[TargetSpec]):
        hooks = []
        for t in targets:
            keep = t.keep_ratio
            if t.component == "head":
                h_idx = t.index
                def head_hook_fn(act, hook, h=h_idx, r=keep):
                    act[:, -1, h, :] = act[:, -1, h, :] * r
                    return act
                hooks.append((f"blocks.{t.layer}.attn.hook_z", head_hook_fn))
            elif t.component == "mlp":
                def mlp_hook_fn(act, hook, r=keep):
                    act[:, -1, :] = act[:, -1, :] * r
                    return act
                hooks.append((f"blocks.{t.layer}.hook_mlp_out", mlp_hook_fn))
        return hooks

    def _resolve_token_id(self, token_str: str) -> int:
        toks = self.model.to_tokens(token_str, prepend_bos=False)[0]
        return int(toks[-1].item())

    @torch.no_grad()
    def measure_grounded_outcomes(self, items: List[PromptItem], targets: Optional[List[TargetSpec]] = None) -> float:
        prompts = [it.prompt for it in items]
        tokens = self.model.to_tokens(prompts)
        batch_size = len(items)
        b_idx = torch.arange(batch_size, device=self.device)

        target_ids = torch.tensor([self._resolve_token_id(it.target_str) for it in items], device=self.device)
        distractor_ids = torch.tensor([self._resolve_token_id(it.distractor_str) for it in items], device=self.device)

        clean_logits = self.model(tokens)[:, -1, :].float()
        clean_m = clean_logits[b_idx, target_ids] - clean_logits[b_idx, distractor_ids]

        if not targets:
            return 0.0

        with self.model.hooks(fwd_hooks=self._build_hooks(targets)):
            int_logits = self.model(tokens)[:, -1, :].float()

        int_m = int_logits[b_idx, target_ids] - int_logits[b_idx, distractor_ids]
        return float(torch.mean(int_m - clean_m).item())


# ==============================================================================
# 4. STRICT BAYESIAN EVoI ACQUISITION ENGINE
# ==============================================================================

class BayesianEVoIEngine:
    def __init__(self, scm: BayesianSelfGraph):
        self.scm = scm
        self.candidates = np.linspace(0.05, 0.40, 8)
        self.eval_shifts = np.array([-2.0, -1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0])
        self.eval_weights = norm.pdf(self.eval_shifts)
        self.eval_weights /= np.sum(self.eval_weights)

    def _evaluate_max_expected_utility(self, mu_star: float, sigma_star: float) -> float:
        sigma_star = max(sigma_star, 1e-6)
        samples = mu_star + sigma_star * self.eval_shifts

        action_expected_utils = []
        for a in EpistemicActionUtility.ALL_ACTIONS:
            eu_a = sum(w * EpistemicActionUtility.get_utility(a, s) for w, s in zip(self.eval_weights, samples))
            action_expected_utils.append(eu_a)

        return float(max(action_expected_utils))

    def select_evoi_dose(self, target: TargetSpec, ctx_str: str, executed_doses: List[float]) -> Tuple[float, bool]:
        self.scm.init_node(target.key, ctx_str)
        node = self.scm.component_models[target.key][extract_context_family(ctx_str)]

        x_full = np.array([1.0, 1.0])
        mu_full = float(x_full @ node["mu"])
        sigma_full = float(np.sqrt(x_full @ node["cov"] @ x_full))

        current_v = self._evaluate_max_expected_utility(mu_full, sigma_full)

        upper_bound = mu_full + 1.645 * sigma_full
        lower_bound = mu_full - 1.645 * sigma_full

        if (upper_bound < EpistemicActionUtility.THRESH_HAZARD or lower_bound >= EpistemicActionUtility.THRESH_GAIN) and len(executed_doses) >= 1:
            return 0.0, True

        best_evoi_score = -float("inf")
        best_dose = self.candidates[0]

        pts, wts = np.polynomial.hermite.hermgauss(5)

        for d in self.candidates:
            if any(abs(d - ed) < 0.03 for ed in executed_doses): continue
            x_d = np.array([d, d**2])
            pred_y = float(x_d @ node["mu"])
            std_y = np.sqrt(float(x_d @ node["cov"] @ x_d) + self.scm.aleatoric_variance)

            prec_sim = node["precision"] + (1.0 / self.scm.aleatoric_variance) * np.outer(x_d, x_d)
            cov_sim = np.linalg.inv(prec_sim)
            sig_full_sim = float(np.sqrt(x_full @ cov_sim @ x_full))

            expected_posterior_v = 0.0
            for pt, w in zip(pts, wts):
                y_sample = pred_y + np.sqrt(2.0) * std_y * pt
                mu_sim = cov_sim @ (node["precision"] @ node["mu"] + (1.0 / self.scm.aleatoric_variance) * x_d * y_sample)
                mu_full_sim = float(x_full @ mu_sim)
                expected_posterior_v += (w / np.sqrt(np.pi)) * self._evaluate_max_expected_utility(mu_full_sim, sig_full_sim)

            evoi = max(0.0, expected_posterior_v - current_v)
            harm = max(0.0, -pred_y)
            barrier = float(np.exp(harm * 40.0) - 1.0)
            score = evoi / (float(d**2) + barrier + 0.05)

            if score > best_evoi_score:
                best_evoi_score = score
                best_dose = float(d)

        if best_evoi_score < 1e-4 and len(executed_doses) >= 1:
            return 0.0, True

        return best_dose, False


# ==============================================================================
# 5. BENCHMARK CONTROLLER (MILESTONE 17.6)
# ==============================================================================

def run_milestone_17_6():
    sensors = MechanisticSensorInterface()

    corpus = {
        "ood_syntactic_agreement": [
            PromptItem("The keys to the cabinet are kept in the", " drawer", " drawers"),
            PromptItem("The books on the wooden table belong to the", " library", " libraries"),
            PromptItem("The pictures of the old palace show the ancient", " king", " kings"),
            PromptItem("The report about the recent discoveries was presented by", " researcher", " researchers"),
        ],
        "ood_distractor_reasoning": [
            PromptItem("While Rome has the Colosseum, the Eiffel Tower stands in", " Paris", " Rome"),
            PromptItem("Even though Madrid is famous, the capital of Germany is", " Berlin", " Madrid"),
            PromptItem("Egypt has pyramids, but the Eiffel Tower was built in the city of", " Paris", " Cairo"),
        ]
    }

    eval_suite = [
        {"desc": "Deep Hazard Node", "target": TargetSpec(20, "mlp", 0, keep_ratio=0.0), "ctx": "ood_syntactic_agreement"},
        {"desc": "Critical Boundary Node", "target": TargetSpec(12, "head", 3, keep_ratio=0.0), "ctx": "ood_syntactic_agreement"},
        {"desc": "Benign Zero-Gain Node", "target": TargetSpec(4, "head", 0, keep_ratio=0.0), "ctx": "ood_syntactic_agreement"},
        {"desc": "Positive Useful Gain", "target": TargetSpec(4, "head", 0, keep_ratio=0.0), "ctx": "ood_distractor_reasoning"}
    ]

    max_budget = 3
    fixed_schedule = [0.10, 0.20, 0.35]

    def resolve_governor(mu: float, sigma: float) -> str:
        if sigma > 0.50: return "VETO_EPISTEMIC"
        if mu < EpistemicActionUtility.THRESH_HAZARD: return "VETO_HAZARD"
        if mu < EpistemicActionUtility.THRESH_GAIN: return "VETO_ZERO_GAIN"
        return "APPROVED"

    print("\n" + "=" * 180)
    print("🔬 MILESTONE 17.6: RIGOROUS BAYESIAN EVoI & GENERALIZED GOVERNANCE BENCHMARK")
    print("   Protocol: E_{y}[max_a EU] - max_a EU Formulation | Regimes: Hazard, Boundary, Zero-Gain, and Approved")
    print("=" * 180)

    results = []

    for item in eval_suite:
        target = item["target"]
        desc = item["desc"]
        ctx = item["ctx"]
        items = corpus[ctx]
        print(f"\n🎯 AUDITING: {target.key} [{desc}] in Context: {ctx}")

        # 1. FIXED POLICY
        scm_fixed = BayesianSelfGraph()
        f_risk = 0.0
        for d in fixed_schedule:
            probe = TargetSpec(target.layer, target.component, target.index, keep_ratio=1.0 - d)
            obs = sensors.measure_grounded_outcomes(items, targets=[probe])
            f_risk += max(0.0, -obs)
            scm_fixed.update(target.key, ctx, d, obs)
        p_f, v_f = scm_fixed.predict([target], ctx)

        # 2. IG-ACTIVE POLICY
        scm_ig = BayesianSelfGraph()
        candidates = np.linspace(0.05, 0.40, 8)
        ig_doses, ig_risk = [], 0.0
        for _ in range(max_budget):
            scm_ig.init_node(target.key, ctx)
            node = scm_ig.component_models[target.key][extract_context_family(ctx)]
            b_sc, b_d = -float("inf"), candidates[0]
            for d in candidates:
                if any(abs(d - ed) < 0.03 for ed in ig_doses): continue
                x = np.array([d, d**2])
                prec_sim = node["precision"] + (1.0 / scm_ig.aleatoric_variance) * np.outer(x, x)
                ig = 0.5 * np.log(max(np.linalg.det(prec_sim) / np.linalg.det(node["precision"]), 1e-12))
                harm = max(0.0, -float(x @ node["mu"]))
                sc = ig / (float(d**2) + 2.0 * harm + 0.05)
                if sc > b_sc: b_sc, b_d = sc, float(d)
            ig_doses.append(b_d)
            probe = TargetSpec(target.layer, target.component, target.index, keep_ratio=1.0 - b_d)
            obs = sensors.measure_grounded_outcomes(items, targets=[probe])
            ig_risk += max(0.0, -obs)
            scm_ig.update(target.key, ctx, b_d, obs)
        p_ig, v_ig = scm_ig.predict([target], ctx)

        # 3. BAYESIAN EVoI-ACTIVE POLICY
        scm_evoi = BayesianSelfGraph()
        evoi_engine = BayesianEVoIEngine(scm_evoi)
        evoi_doses, evoi_risk = [], 0.0
        stopped_early = False

        for _ in range(max_budget):
            d, should_stop = evoi_engine.select_evoi_dose(target, ctx, evoi_doses)
            if should_stop:
                stopped_early = True
                break
            evoi_doses.append(d)
            probe = TargetSpec(target.layer, target.component, target.index, keep_ratio=1.0 - d)
            obs = sensors.measure_grounded_outcomes(items, targets=[probe])
            evoi_risk += max(0.0, -obs)
            scm_evoi.update(target.key, ctx, d, obs)
        p_evoi, v_evoi = scm_evoi.predict([target], ctx)

        # HIDDEN VALIDATION
        gt_delta = sensors.measure_grounded_outcomes(items, targets=[target])
        oracle_action, oracle_u = EpistemicActionUtility.get_omniscient_oracle(gt_delta)

        def pack_res(pred_m, sig_m, risk, doses, stopped=False):
            verdict = resolve_governor(pred_m, sig_m)
            u = EpistemicActionUtility.get_utility(verdict, gt_delta)
            regret = oracle_u - u
            iei = (u + 0.05) / (risk + 0.01)
            return {
                "doses": doses,
                "steps": len(doses),
                "pred": pred_m,
                "sigma": sig_m,
                "mae": abs(pred_m - gt_delta),
                "verdict": verdict,
                "utility": u,
                "regret": regret,
                "risk": risk,
                "iei": iei,
                "early_stop": stopped
            }

        results.append({
            "target": target.key,
            "desc": desc,
            "ctx": ctx,
            "gt_delta": gt_delta,
            "oracle_action": oracle_action,
            "oracle_u": oracle_u,
            "fixed": pack_res(p_f, float(np.sqrt(v_f)), f_risk, fixed_schedule),
            "ig": pack_res(p_ig, float(np.sqrt(v_ig)), ig_risk, [round(x, 2) for x in ig_doses]),
            "evoi": pack_res(p_evoi, float(np.sqrt(v_evoi)), evoi_risk, [round(x, 2) for x in evoi_doses], stopped_early)
        })

    # ==============================================================================
    # 6. AUDIT REPORT TABLE
    # ==============================================================================
    print("\n" + "=" * 180)
    print("📊 BENCHMARK 17.6 COMPARATIVE AUDIT REPORT")
    print("-" * 180)
    print(f"{'Target Key':<12} | {'Policy':<14} | {'Dose Sequence':<23} | {'Steps':<5} | {'Risk Incurred':<14} | {'Verdict':<16} | {'Regret':<8} | {'IEI (Efficiency)'}")
    print("-" * 180)

    for r in results:
        t_key = r["target"]
        print(f"[{t_key}] ({r['desc']}) -> Oracle: {r['oracle_action']}, GT ΔMc: {r['gt_delta']:+.4f}")

        f = r["fixed"]
        print(f"{'':<12} | {'FIXED':<14} | {str(f['doses']):<23} | {f['steps']:<5} | {f['risk']:<14.4f} | {f['verdict']:<16} | {f['regret']:.4f}   | {f['iei']:.2f}")

        ig = r["ig"]
        print(f"{'':<12} | {'IG-ACTIVE':<14} | {str(ig['doses']):<23} | {ig['steps']:<5} | {ig['risk']:<14.4f} | {ig['verdict']:<16} | {ig['regret']:.4f}   | {ig['iei']:.2f}")

        ev = r["evoi"]
        tag = " (EARLY_STOP)" if ev["early_stop"] else ""
        print(f"{'':<12} | {'EVoI-ACTIVE':<14} | {str(ev['doses']) + tag:<23} | {ev['steps']:<5} | {ev['risk']:<14.4f} | {ev['verdict']:<16} | {ev['regret']:.4f}   | {ev['iei']:.2f}")
        print("-" * 180)

    with open("m17_6_generalization_results.json", "w", encoding="utf-8") as f:
        json.dump(to_jsonable(results), f, indent=2)
    print("📁 Benchmark data saved to 'm17_6_generalization_results.json'.")


if __name__ == "__main__":
    run_milestone_17_6()

🔄 Initializing Sensors (Qwen/Qwen2.5-0.5B) on CUDA...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]